## Data Enrichment

### Loading Libraries

In [1]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [2]:

citibike_df = pd.read_csv("../data/citibike/JC/JC2025.csv")

C:\Users\User\AppData\Local\Temp\ipykernel_15936\1730691477.py:1: DtypeWarning: Columns (14,15,16,18,20) have mixed types. Specify dtype option on import or set low_memory=False.
  citibike_df = pd.read_csv("../data/citibike/JC/JC2025.csv")


###  Observe Missing Values

In [3]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
16,month_name,3808372,0.692527
17,month_number,3808372,0.692527
18,day_of_week,3808372,0.692527
19,hour,3808372,0.692527
20,season,3808372,0.692527
14,date,3808372,0.692527
15,month,3808372,0.692527
13,ride_duration_min,3808372,0.692527
7,end_station_id,17060,0.003102
11,end_lng,13700,0.002491


In [4]:
citibike_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5499236 entries, 0 to 5499235
Data columns (total 21 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
 13  ride_duration_min   float64
 14  date                object 
 15  month               object 
 16  month_name          object 
 17  month_number        float64
 18  day_of_week         object 
 19  hour                float64
 20  season              object 
dtypes: float64(7), object(14)
memory usage: 881.1+ MB


### Duration

In [5]:
# Convert the text columns into actual datetime objects
citibike_df["started_at"] = pd.to_datetime(citibike_df["started_at"])
citibike_df["ended_at"] = pd.to_datetime(citibike_df["ended_at"])

In [6]:
citibike_df['ride_duration_min'] = (citibike_df["ended_at"] - citibike_df["started_at"]).dt.total_seconds()/60

In [7]:
citibike_df = citibike_df[
    (citibike_df['ride_duration_min']>1) & (citibike_df['ride_duration_min']<24*60)
]

citibike_df.shape

(5497836, 21)

In [8]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'ride_duration_min', 'date', 'month', 'month_name',
       'month_number', 'day_of_week', 'hour', 'season'],
      dtype='object')

In [9]:
citibike_df = citibike_df.dropna(
    subset=['ride_id', 'started_at', 'ended_at','start_station_name', 'start_station_id', 'end_station_name',
    'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
        ]
)

### Time Based Variables

In [10]:
citibike_df['date'] = citibike_df['started_at'].dt.date
citibike_df['month'] = citibike_df['started_at'].dt.to_period('M').astype(str)
citibike_df['month_name'] = citibike_df['started_at'].dt.month_name()
citibike_df['month_number'] = citibike_df['started_at'].dt.month
citibike_df['day_of_week'] = citibike_df['started_at'].dt.day_name()
citibike_df['hour'] = citibike_df['started_at'].dt.hour

In [11]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,ride_duration_min,date,month,month_name,month_number,day_of_week,hour,season
0,04CF7A399050E404,classic_bike,2025-02-22 17:40:16.500,2025-02-22 17:47:22.479,Jersey & 3rd,JC074,Van Vorst Park,JC035,40.723332,-74.045953,...,-74.047727,casual,7.09965,2025-02-22,2025-02,February,2,Saturday,17,NaN
1,124AC7493E82D845,classic_bike,2025-02-21 12:28:13.319,2025-02-21 12:35:44.762,Jersey & 3rd,JC074,Columbus Drive,JC014,40.723332,-74.045953,...,-74.038914,member,7.52405,2025-02-21,2025-02,February,2,Friday,12,NaN
2,1A3BCA838E968327,classic_bike,2025-02-01 14:17:43.272,2025-02-01 14:59:09.894,Jersey & 3rd,JC074,Grove St PATH,JC115,40.723332,-74.045953,...,-74.043090,casual,41.44370,2025-02-01,2025-02,February,2,Saturday,14,NaN
3,5994017EE989D6EE,electric_bike,2025-02-22 11:36:29.292,2025-02-22 11:49:51.531,Jersey & 3rd,JC074,Jersey & 3rd,JC074,40.723332,-74.045953,...,-74.045953,casual,13.37065,2025-02-22,2025-02,February,2,Saturday,11,NaN
4,F81BCB97915C6BE6,electric_bike,2025-02-28 22:56:26.546,2025-02-28 23:07:40.391,Jersey & 3rd,JC074,Monroe St & 11 St,HB508,40.723332,-74.045953,...,-74.036637,casual,11.23075,2025-02-28,2025-02,February,2,Friday,22,NaN


In [12]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [13]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-02-22 17:40:16.500,2025-02-22,2025-02,February,Saturday,17,Winter
1,2025-02-21 12:28:13.319,2025-02-21,2025-02,February,Friday,12,Winter
2,2025-02-01 14:17:43.272,2025-02-01,2025-02,February,Saturday,14,Winter
3,2025-02-22 11:36:29.292,2025-02-22,2025-02,February,Saturday,11,Winter
4,2025-02-28 22:56:26.546,2025-02-28,2025-02,February,Friday,22,Winter


In [14]:
citibike_df.to_csv("../data/citibike/JC/JC2025_Enriched.csv", index = False)